# Vélib Disponibilité en Temps Réel - Bronze Layer

## Objective

Read the raw Vélib availability JSON file stored in the Databricks Volume and create the Bronze layer as a Delta table.

## Data Flow

Raw JSON in Databricks Volume → Spark DataFrame → Bronze Delta Table

## Source

The data comes from the Paris OpenData dataset:

**Vélib - Disponibilité en temps réel**

The raw data was previously retrieved from the Paris OpenData API and stored in the Databricks Volume.

## Input

Raw JSON file:

/Volumes/workspace/default/raw_data/velib_disponibilite_raw.json

## Output

Bronze Delta table:

bronze_velib

## Bronze Layer Principle

The Bronze layer keeps the source data as close as possible to the original data.

No business transformations or data cleaning are performed at this stage.

The original column order is preserved, and a technical ingestion timestamp is added to track when the data was loaded into the Bronze layer.

## Processing Steps

1. Read the raw JSON file from the Databricks Volume.
2. Retrieve the original column order from the JSON file.
3. Create a Spark DataFrame.
4. Preserve the original column order.
5. Add _ingestion_timestamp.
6. Write the data as a Delta table named bronze_velib.
7. Display the Bronze table for verification.

In [0]:
# importing libraries 
from pyspark.sql.functions import current_timestamp
import json

In [0]:
# 1. Path to raw JSON in Volume
RAW_PATH = "/Volumes/workspace/default/raw_data/velib_disponibilite_raw.json"

# 2. Read raw JSON
df = spark.read.option("multiline", "true").json(RAW_PATH)

# 3. Keep original column order
original_column_order = [
    "stationcode",
    "name",
    "is_installed",
    "capacity",
    "numdocksavailable",
    "numbikesavailable",
    "mechanical",
    "ebike",
    "is_renting",
    "is_returning",
    "duedate",
    "coordonnees_geo",
    "nom_arrondissement_communes",
    "code_insee_commune",
    "station_opening_hours"
]

# Keep only columns that exist in the DataFrame
original_column_order = [
    col for col in original_column_order if col in df.columns
]

df = df.select(*original_column_order)

# 4. Add Bronze metadata
df_bronze = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
)

# 5. Write Bronze Delta table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.bronze_velib")

# 6. Display Bronze table
display(df_bronze)